# 06 Engagement Analysis

This notebook applies the best-performing AlexNet binary sentiment classifier to Instagram brand posts and examines how predicted image sentiment relates to engagement outcomes such as likes and comments. It includes regression models and binary high-vs-low engagement classification.

In [ ]:
from google.colab import drive
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

In [ ]:
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import torchvision.models as models
import torch.optim as optim
import torch.nn as nn
from tqdm import tqdm

In [ ]:
PATH_TL    = "/content/drive/MyDrive/alexnet_transfer_best_2classes.pth"  # ← update if needed
num_epochs = 30  # increase to 30-50 for a proper training run

In [ ]:
# Load ImageNet pretrained weights
builtin_alexnet = models.alexnet(weights=models.AlexNet_Weights.IMAGENET1K_V1)

# Step 1 — freeze all layers
for param in builtin_alexnet.parameters():
    param.requires_grad = False

# Step 2 — unfreeze the last conv block (features[10-12])
for param in builtin_alexnet.features[10:].parameters():
    param.requires_grad = True

# Step 3 — replace classifier head for 4 sentiment classes
builtin_alexnet.classifier[6] = nn.Linear(in_features=4096, out_features=2)

# Move to device
builtin_alexnet = builtin_alexnet.to(device)

Downloading: "https://download.pytorch.org/models/alexnet-owt-7be5be79.pth" to /root/.cache/torch/hub/checkpoints/alexnet-owt-7be5be79.pth


100%|██████████| 233M/233M [00:00<00:00, 268MB/s]


Predicting Sentiment of Social Media Posts Using the Best Trained Model from Above

In [ ]:
# Load the best saved model
builtin_alexnet.load_state_dict(torch.load(PATH_TL))
builtin_alexnet.eval()

AlexNet(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(11, 11), stride=(4, 4), padding=(2, 2))
    (1): ReLU(inplace=True)
    (2): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(64, 192, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
    (4): ReLU(inplace=True)
    (5): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(192, 384, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU(inplace=True)
    (8): Conv2d(384, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): ReLU(inplace=True)
    (10): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (avgpool): AdaptiveAvgPool2d(output_size=(6, 6))
  (classifier): Sequential(
    (0): Dropout(p=0.5, inplace=False)
    (1): Linear(in_features=9216, out_features=4096, bias=True)
 

In [ ]:
import os
from PIL import Image
import torch
from torchvision import datasets

# 1. CONFIG
NEW_DATA_DIR = "/content/drive/MyDrive/Social Media Dataset/Images"  # ← update this path
RESULTS_PATH = "/content/drive/MyDrive/Social Media Dataset/predictions.csv"   # ← where to save results

# 2. LOAD BEST MODEL
builtin_alexnet.load_state_dict(torch.load(PATH_TL))
builtin_alexnet.eval()

# 3. SAME TRANSFORMS AS VAL/TEST
inference_transforms = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# 4. RUN PREDICTIONS
label_names = {0: "negative", 1: "positive"}
results     = []

image_files = [f for f in os.listdir(NEW_DATA_DIR) if f.endswith(".png")]
print(f"Found {len(image_files)} images — running predictions...\n")

for fname in sorted(image_files):
    path  = os.path.join(NEW_DATA_DIR, fname)
    image = Image.open(path).convert("RGB")
    tensor = inference_transforms(image).unsqueeze(0).to(device)

    with torch.no_grad():
        output    = builtin_alexnet(tensor)
        probs     = torch.softmax(output, dim=1)
        pred      = output.argmax(dim=1).item()
        confidence = probs[0][pred].item() * 100

    results.append((fname, label_names[pred], f"{confidence:.2f}%"))
    print(f"{fname:<30} → {label_names[pred]:<10} (confidence: {confidence:.2f}%)")

# 5. SAVE RESULTS TO CSV
import csv
import pandas as pd

with open(RESULTS_PATH, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["filename", "prediction", "confidence"])
    writer.writerows(results)

df_results = pd.DataFrame(results, columns=["filename", "prediction", "confidence"])
print(df_results)
print(f"\nSaved to {RESULTS_PATH}")

# 6. SUMMARY
positive_count = sum(1 for _, label, _ in results if label == "positive")
negative_count = sum(1 for _, label, _ in results if label == "negative")
print(f"\nSummary — Positive: {positive_count} | Negative: {negative_count} | Total: {len(results)}")

Found 100 images — running predictions...

ABC 1.png                      → positive   (confidence: 99.99%)
ABC 10.png                     → positive   (confidence: 50.88%)
ABC 2.png                      → negative   (confidence: 100.00%)
ABC 3.png                      → negative   (confidence: 61.55%)
ABC 4.png                      → positive   (confidence: 88.06%)
ABC 5.png                      → negative   (confidence: 100.00%)
ABC 6.png                      → positive   (confidence: 98.64%)
ABC 7.png                      → negative   (confidence: 88.27%)
ABC 8.png                      → positive   (confidence: 96.12%)
ABC 9.png                      → negative   (confidence: 99.82%)
BBC 1.png                      → negative   (confidence: 100.00%)
BBC 10.png                     → negative   (confidence: 99.93%)
BBC 2.png                      → negative   (confidence: 99.86%)
BBC 3.png                      → positive   (confidence: 98.89%)
BBC 4.png                      → negative   

In [ ]:
import pandas as pd

# Load your other CSV
df_other = pd.read_csv("/content/drive/MyDrive/Social Media Dataset/Image Data.csv")

# Rename filename column to match your other CSV
df_results = df_results.rename(columns={"filename": "Image Name"})

# Merge on Image Name
df_merged = pd.merge(df_other, df_results, on="Image Name")

# Convert prediction to numeric for regression
# positive = 1, negative = 0
df_merged["sentiment_score"] = df_merged["prediction"].map({"positive": 1, "negative": 0})

# Save merged file
df_merged.to_csv("/content/drive/MyDrive/Social Media Dataset/Merged_Data.csv", index=False)

# Preview
print(f"Merged dataset shape: {df_merged.shape}")
print(df_merged.head())

Merged dataset shape: (99, 12)
    Image Name  # Likes  # Comments  # Followers  # Following  # Posts  \
0  image_1.png    30400         127     20000000          626    23745   
1  image_2.png   223000        4120     20000000          626    23745   
2  image_3.png    14800          79     20000000          626    23745   
3  image_4.png     6800          39     20000000          626    23745   
4  image_5.png    21000         373     20000000          626    23745   

   Multiple Image Indicator  Day Posted  Engagement Calculation prediction  \
0                      True    20260406                0.001526   negative   
1                     False    20260403                0.011356   positive   
2                      True    20260329                0.000744   negative   
3                     False    20260328                0.000342   positive   
4                      True    20260325                0.001069   negative   

  confidence  sentiment_score  
0     99.65%           

In [ ]:
df_merged.to_csv("/content/drive/MyDrive/merged_data.csv", index=False)
print(df_merged.head())

    Image Name  # Likes  # Comments  # Followers  # Following  # Posts  \
0  image_1.png    30400         127     20000000          626    23745   
1  image_2.png   223000        4120     20000000          626    23745   
2  image_3.png    14800          79     20000000          626    23745   
3  image_4.png     6800          39     20000000          626    23745   
4  image_5.png    21000         373     20000000          626    23745   

   Multiple Image Indicator  Day Posted  Engagement Calculation prediction  \
0                      True    20260406                0.001526   negative   
1                     False    20260403                0.011356   positive   
2                      True    20260329                0.000744   negative   
3                     False    20260328                0.000342   positive   
4                      True    20260325                0.001069   negative   

  confidence  sentiment_score  
0     99.65%                0  
1     99.84%          

Linear Regression to predict engagement

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error
import pandas as pd
import numpy as np

# 1. FEATURE ENGINEERING
df = df_merged.copy()

# Convert Day Posted to datetime and extract components
df["Day Posted"]  = pd.to_datetime(df["Day Posted"], format="%Y%m%d")
df["day_of_week"] = df["Day Posted"].dt.dayofweek   # 0=Monday, 6=Sunday
df["month"]       = df["Day Posted"].dt.month        # 1-12
df["year"]        = df["Day Posted"].dt.year         # e.g. 2026

# Convert Multiple Image Indicator to numeric
df["multiple_image"] = df["Multiple Image Indicator"].astype(int)

# Convert confidence to numeric (remove % sign)
df["confidence"] = df["confidence"].str.replace("%", "").astype(float)

# 2. DEFINE FEATURES AND TARGET
X = df[[
    "sentiment_score",
    "multiple_image",
    "day_of_week",
    "month",
    "year",
    "# Following",
    "# Posts"
]]

y = df["Engagement Calculation"]

# 3. TRAIN / TEST SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 4. FIT REGRESSION
model = LinearRegression()
model.fit(X_train, y_train)

# 5. EVALUATE
y_pred = model.predict(X_test)
r2     = r2_score(y_test, y_pred)
rmse   = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"R² Score : {r2:.4f}")
print(f"RMSE     : {rmse:.6f}")

# 6. COEFFICIENTS — shows impact of each variable
coef_df = pd.DataFrame({
    "Feature"    : X.columns,
    "Coefficient": model.coef_
}).sort_values("Coefficient", ascending=False)

print("\nFeature Coefficients:")
print(coef_df.to_string(index=False))
print(f"\nIntercept: {model.intercept_:.6f}")

R² Score : -0.6288
RMSE     : 0.004363

Feature Coefficients:
        Feature   Coefficient
sentiment_score  1.916854e-03
 multiple_image  1.211694e-03
    # Following  3.985673e-06
        # Posts -1.435914e-09
    day_of_week -2.488928e-04
          month -4.352929e-04
           year -2.139955e-03

Intercept: 4.338135


In [ ]:
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score
import numpy as np

# Use cross-validation instead of a single
# train/test split — more reliable with 99 rows
models = {
    "Linear Regression": LinearRegression(),
    "Ridge"            : Ridge(alpha=1.0),
    "Lasso"            : Lasso(alpha=0.0001),
}

print("5-Fold Cross Validation Results:")
print("-" * 45)

for name, reg_model in models.items():
    # Scale features — important for Ridge and Lasso
    pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("model",  reg_model)
    ])

    cv_r2   = cross_val_score(pipeline, X, y, cv=5, scoring="r2")
    cv_rmse = cross_val_score(pipeline, X, y, cv=5,
                               scoring="neg_root_mean_squared_error")

    print(f"\n{name}")
    print(f"  R²   : {cv_r2.mean():.4f} (± {cv_r2.std():.4f})")
    print(f"  RMSE : {-cv_rmse.mean():.6f} (± {cv_rmse.std():.6f})")

# Coefficient comparison across models
print("\n\nCoefficient Comparison (scaled features):")
print("-" * 55)

for name, reg_model in models.items():
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    reg_model.fit(X_scaled, y)

    coef_df = pd.DataFrame({
        "Feature"    : X.columns,
        "Coefficient": reg_model.coef_
    }).sort_values("Coefficient", key=abs, ascending=False)

    print(f"\n{name}:")
    print(coef_df.to_string(index=False))

5-Fold Cross Validation Results:
---------------------------------------------

Linear Regression
  R²   : -0.3550 (± 0.3002)
  RMSE : 0.004123 (± 0.001138)

Ridge
  R²   : -0.3179 (± 0.2731)
  RMSE : 0.004072 (± 0.001131)

Lasso
  R²   : -0.1670 (± 0.2093)
  RMSE : 0.003850 (± 0.001123)


Coefficient Comparison (scaled features):
-------------------------------------------------------

Linear Regression:
        Feature  Coefficient
sentiment_score     0.000563
          month    -0.000324
    # Following     0.000314
 multiple_image     0.000313
           year    -0.000136
    day_of_week     0.000082
        # Posts     0.000041

Ridge:
        Feature  Coefficient
sentiment_score     0.000559
    # Following     0.000310
 multiple_image     0.000309
          month    -0.000308
           year    -0.000120
    day_of_week     0.000083
        # Posts     0.000037

Lasso:
        Feature  Coefficient
sentiment_score     0.000501
    # Following     0.000241
 multiple_image     0.00

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error
import pandas as pd
import numpy as np

# 1. FEATURE ENGINEERING
df = df_merged.copy()

# Convert Day Posted to datetime and extract components
df["Day Posted"]  = pd.to_datetime(df["Day Posted"], format="%Y%m%d")
df["day_of_week"] = df["Day Posted"].dt.dayofweek   # 0=Monday, 6=Sunday
df["month"]       = df["Day Posted"].dt.month        # 1-12
df["year"]        = df["Day Posted"].dt.year         # e.g. 2026

# Convert Multiple Image Indicator to numeric
df["multiple_image"] = df["Multiple Image Indicator"].astype(int)

# Convert confidence to numeric (remove % sign)
df["confidence"] = df["confidence"].str.replace("%", "").astype(float)

# 2. DEFINE FEATURES AND TARGET
X = df[[
    "sentiment_score",
    "multiple_image",
    "day_of_week",
    "# Following"
]]

y = df["Engagement Calculation"]

# 3. TRAIN / TEST SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 4. FIT REGRESSION
model = LinearRegression()
model.fit(X_train, y_train)

# 5. EVALUATE
y_pred = model.predict(X_test)
r2     = r2_score(y_test, y_pred)
rmse   = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"R² Score : {r2:.4f}")
print(f"RMSE     : {rmse:.6f}")

# 6. COEFFICIENTS — shows impact of each variable
coef_df = pd.DataFrame({
    "Feature"    : X.columns,
    "Coefficient": model.coef_
}).sort_values("Coefficient", ascending=False)

print("\nFeature Coefficients:")
print(coef_df.to_string(index=False))
print(f"\nIntercept: {model.intercept_:.6f}")

R² Score : -0.4033
RMSE     : 0.004049

Feature Coefficients:
        Feature  Coefficient
sentiment_score     0.001889
 multiple_image     0.001181
    # Following     0.000004
    day_of_week    -0.000226

Intercept: 0.001232


In [ ]:
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score
import numpy as np

# Use cross-validation instead of a single
# train/test split — more reliable with 99 rows
models = {
    "Linear Regression": LinearRegression(),
    "Ridge"            : Ridge(alpha=1.0),
    "Lasso"            : Lasso(alpha=0.0001),
}

print("5-Fold Cross Validation Results:")
print("-" * 45)

for name, reg_model in models.items():
    # Scale features — important for Ridge and Lasso
    pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("model",  reg_model)
    ])

    cv_r2   = cross_val_score(pipeline, X, y, cv=5, scoring="r2")
    cv_rmse = cross_val_score(pipeline, X, y, cv=5,
                               scoring="neg_root_mean_squared_error")

    print(f"\n{name}")
    print(f"  R²   : {cv_r2.mean():.4f} (± {cv_r2.std():.4f})")
    print(f"  RMSE : {-cv_rmse.mean():.6f} (± {cv_rmse.std():.6f})")

# Coefficient comparison across models
print("\n\nCoefficient Comparison (scaled features):")
print("-" * 55)

for name, reg_model in models.items():
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    reg_model.fit(X_scaled, y)

    coef_df = pd.DataFrame({
        "Feature"    : X.columns,
        "Coefficient": reg_model.coef_
    }).sort_values("Coefficient", key=abs, ascending=False)

    print(f"\n{name}:")
    print(coef_df.to_string(index=False))

5-Fold Cross Validation Results:
---------------------------------------------

Linear Regression
  R²   : -0.0909 (± 0.1941)
  RMSE : 0.003745 (± 0.001163)

Ridge
  R²   : -0.0885 (± 0.1910)
  RMSE : 0.003741 (± 0.001161)

Lasso
  R²   : -0.0644 (± 0.1533)
  RMSE : 0.003699 (± 0.001131)


Coefficient Comparison (scaled features):
-------------------------------------------------------

Linear Regression:
        Feature  Coefficient
sentiment_score     0.000591
    # Following     0.000331
 multiple_image     0.000289
    day_of_week     0.000092

Ridge:
        Feature  Coefficient
sentiment_score     0.000586
    # Following     0.000328
 multiple_image     0.000287
    day_of_week     0.000092

Lasso:
        Feature  Coefficient
sentiment_score     0.000510
    # Following     0.000261
 multiple_image     0.000215
    day_of_week     0.000030


In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error
import pandas as pd
import numpy as np

# 1. FEATURE ENGINEERING
df = df_merged.copy()

# Convert Day Posted to datetime and extract components
df["Day Posted"]  = pd.to_datetime(df["Day Posted"], format="%Y%m%d")
df["day_of_week"] = df["Day Posted"].dt.dayofweek   # 0=Monday, 6=Sunday
df["month"]       = df["Day Posted"].dt.month        # 1-12
df["year"]        = df["Day Posted"].dt.year         # e.g. 2026

# Convert Multiple Image Indicator to numeric
df["multiple_image"] = df["Multiple Image Indicator"].astype(int)

# Convert confidence to numeric (remove % sign)
df["confidence"] = df["confidence"].str.replace("%", "").astype(float)

# 2. DEFINE FEATURES AND TARGET
X = df[[
    "sentiment_score"
]]

y = df["Engagement Calculation"]

# 3. TRAIN / TEST SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 4. FIT REGRESSION
model = LinearRegression()
model.fit(X_train, y_train)

# 5. EVALUATE
y_pred = model.predict(X_test)
r2     = r2_score(y_test, y_pred)
rmse   = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"R² Score : {r2:.4f}")
print(f"RMSE     : {rmse:.6f}")

# 6. COEFFICIENTS — shows impact of each variable
coef_df = pd.DataFrame({
    "Feature"    : X.columns,
    "Coefficient": model.coef_
}).sort_values("Coefficient", ascending=False)

print("\nFeature Coefficients:")
print(coef_df.to_string(index=False))
print(f"\nIntercept: {model.intercept_:.6f}")

R² Score : -0.1053
RMSE     : 0.003594

Feature Coefficients:
        Feature  Coefficient
sentiment_score     0.002094

Intercept: 0.002000


In [ ]:
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score
import numpy as np

# Use cross-validation instead of a single
# train/test split — more reliable with 99 rows
models = {
    "Linear Regression": LinearRegression(),
    "Ridge"            : Ridge(alpha=1.0),
    "Lasso"            : Lasso(alpha=0.0001),
}

print("5-Fold Cross Validation Results:")
print("-" * 45)

for name, reg_model in models.items():
    # Scale features — important for Ridge and Lasso
    pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("model",  reg_model)
    ])

    cv_r2   = cross_val_score(pipeline, X, y, cv=5, scoring="r2")
    cv_rmse = cross_val_score(pipeline, X, y, cv=5,
                               scoring="neg_root_mean_squared_error")

    print(f"\n{name}")
    print(f"  R²   : {cv_r2.mean():.4f} (± {cv_r2.std():.4f})")
    print(f"  RMSE : {-cv_rmse.mean():.6f} (± {cv_rmse.std():.6f})")

# Coefficient comparison across models
print("\n\nCoefficient Comparison (scaled features):")
print("-" * 55)

for name, reg_model in models.items():
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    reg_model.fit(X_scaled, y)

    coef_df = pd.DataFrame({
        "Feature"    : X.columns,
        "Coefficient": reg_model.coef_
    }).sort_values("Coefficient", key=abs, ascending=False)

    print(f"\n{name}:")
    print(coef_df.to_string(index=False))

5-Fold Cross Validation Results:
---------------------------------------------

Linear Regression
  R²   : -0.0052 (± 0.0805)
  RMSE : 0.003589 (± 0.001047)

Ridge
  R²   : -0.0049 (± 0.0793)
  RMSE : 0.003588 (± 0.001046)

Lasso
  R²   : -0.0059 (± 0.0676)
  RMSE : 0.003591 (± 0.001049)


Coefficient Comparison (scaled features):
-------------------------------------------------------

Linear Regression:
        Feature  Coefficient
sentiment_score     0.000644

Ridge:
        Feature  Coefficient
sentiment_score     0.000637

Lasso:
        Feature  Coefficient
sentiment_score     0.000544


Predicting # Comments

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error
import pandas as pd
import numpy as np

# 1. FEATURE ENGINEERING
df = df_merged.copy()

# Convert Day Posted to datetime and extract components
df["Day Posted"]  = pd.to_datetime(df["Day Posted"], format="%Y%m%d")
df["day_of_week"] = df["Day Posted"].dt.dayofweek   # 0=Monday, 6=Sunday
df["month"]       = df["Day Posted"].dt.month        # 1-12
df["year"]        = df["Day Posted"].dt.year         # e.g. 2026

# Convert Multiple Image Indicator to numeric
df["multiple_image"] = df["Multiple Image Indicator"].astype(int)

# Convert confidence to numeric (remove % sign)
df["confidence"] = df["confidence"].str.replace("%", "").astype(float)

# 2. DEFINE FEATURES AND TARGET
X = df[[
    "sentiment_score",
    "multiple_image",
    "day_of_week",
    "month",
    "year",
    "# Following",
    "# Posts"
]]

y = df["# Comments"]

# 3. TRAIN / TEST SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 4. FIT REGRESSION
model = LinearRegression()
model.fit(X_train, y_train)

# 5. EVALUATE
y_pred = model.predict(X_test)
r2     = r2_score(y_test, y_pred)
rmse   = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"R² Score : {r2:.4f}")
print(f"RMSE     : {rmse:.6f}")

# 6. COEFFICIENTS — shows impact of each variable
coef_df = pd.DataFrame({
    "Feature"    : X.columns,
    "Coefficient": model.coef_
}).sort_values("Coefficient", ascending=False)

print("\nFeature Coefficients:")
print(coef_df.to_string(index=False))
print(f"\nIntercept: {model.intercept_:.6f}")

R² Score : 0.0038
RMSE     : 3275.438617

Feature Coefficients:
        Feature  Coefficient
           year   640.871353
 multiple_image   308.616996
          month   122.147461
sentiment_score    88.732825
    # Following     0.364367
        # Posts     0.014064
    day_of_week   -29.634189

Intercept: -1298210.790849


In [ ]:
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score
import numpy as np

# Use cross-validation instead of a single
# train/test split — more reliable with 99 rows
models = {
    "Linear Regression": LinearRegression(),
    "Ridge"            : Ridge(alpha=1.0),
    "Lasso"            : Lasso(alpha=0.0001),
}

print("5-Fold Cross Validation Results:")
print("-" * 45)

for name, reg_model in models.items():
    # Scale features — important for Ridge and Lasso
    pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("model",  reg_model)
    ])

    cv_r2   = cross_val_score(pipeline, X, y, cv=5, scoring="r2")
    cv_rmse = cross_val_score(pipeline, X, y, cv=5,
                               scoring="neg_root_mean_squared_error")

    print(f"\n{name}")
    print(f"  R²   : {cv_r2.mean():.4f} (± {cv_r2.std():.4f})")
    print(f"  RMSE : {-cv_rmse.mean():.6f} (± {cv_rmse.std():.6f})")

# Coefficient comparison across models
print("\n\nCoefficient Comparison (scaled features):")
print("-" * 55)

for name, reg_model in models.items():
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    reg_model.fit(X_scaled, y)

    coef_df = pd.DataFrame({
        "Feature"    : X.columns,
        "Coefficient": reg_model.coef_
    }).sort_values("Coefficient", key=abs, ascending=False)

    print(f"\n{name}:")
    print(coef_df.to_string(index=False))

5-Fold Cross Validation Results:
---------------------------------------------

Linear Regression
  R²   : -2.6202 (± 3.2142)
  RMSE : 2516.560047 (± 1423.114365)

Ridge
  R²   : -2.6598 (± 3.1767)
  RMSE : 2517.654565 (± 1412.597616)

Lasso
  R²   : -2.6202 (± 3.2142)
  RMSE : 2516.560240 (± 1423.113905)


Coefficient Comparison (scaled features):
-------------------------------------------------------

Linear Regression:
        Feature  Coefficient
           year   415.755953
          month   330.136186
        # Posts   324.004069
    day_of_week   187.477355
 multiple_image   180.514907
    # Following   -90.868466
sentiment_score   -21.618528

Ridge:
        Feature  Coefficient
           year   391.249756
        # Posts   325.828606
          month   306.955135
    day_of_week   185.034162
 multiple_image   179.056075
    # Following   -88.144109
sentiment_score   -22.992823

Lasso:
        Feature  Coefficient
           year   415.755275
          month   330.135548
      

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error
import pandas as pd
import numpy as np

# 1. FEATURE ENGINEERING
df = df_merged.copy()

# Convert Day Posted to datetime and extract components
df["Day Posted"]  = pd.to_datetime(df["Day Posted"], format="%Y%m%d")
df["day_of_week"] = df["Day Posted"].dt.dayofweek   # 0=Monday, 6=Sunday
df["month"]       = df["Day Posted"].dt.month        # 1-12
df["year"]        = df["Day Posted"].dt.year         # e.g. 2026

# Convert Multiple Image Indicator to numeric
df["multiple_image"] = df["Multiple Image Indicator"].astype(int)

# Convert confidence to numeric (remove % sign)
df["confidence"] = df["confidence"].str.replace("%", "").astype(float)

# 2. DEFINE FEATURES AND TARGET
X = df[[
    "sentiment_score",
    "multiple_image",
    "day_of_week",
    "# Following"
]]

y = df["# Comments"]

# 3. TRAIN / TEST SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 4. FIT REGRESSION
model = LinearRegression()
model.fit(X_train, y_train)

# 5. EVALUATE
y_pred = model.predict(X_test)
r2     = r2_score(y_test, y_pred)
rmse   = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"R² Score : {r2:.4f}")
print(f"RMSE     : {rmse:.6f}")

# 6. COEFFICIENTS — shows impact of each variable
coef_df = pd.DataFrame({
    "Feature"    : X.columns,
    "Coefficient": model.coef_
}).sort_values("Coefficient", ascending=False)

print("\nFeature Coefficients:")
print(coef_df.to_string(index=False))
print(f"\nIntercept: {model.intercept_:.6f}")

R² Score : -0.0354
RMSE     : 3339.318180

Feature Coefficients:
        Feature  Coefficient
sentiment_score   258.641184
 multiple_image   189.682431
    # Following    -0.086630
    day_of_week   -24.762564

Intercept: 1106.646639


In [ ]:
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score
import numpy as np

# Use cross-validation instead of a single
# train/test split — more reliable with 99 rows
models = {
    "Linear Regression": LinearRegression(),
    "Ridge"            : Ridge(alpha=1.0),
    "Lasso"            : Lasso(alpha=0.0001),
}

print("5-Fold Cross Validation Results:")
print("-" * 45)

for name, reg_model in models.items():
    # Scale features — important for Ridge and Lasso
    pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("model",  reg_model)
    ])

    cv_r2   = cross_val_score(pipeline, X, y, cv=5, scoring="r2")
    cv_rmse = cross_val_score(pipeline, X, y, cv=5,
                               scoring="neg_root_mean_squared_error")

    print(f"\n{name}")
    print(f"  R²   : {cv_r2.mean():.4f} (± {cv_r2.std():.4f})")
    print(f"  RMSE : {-cv_rmse.mean():.6f} (± {cv_rmse.std():.6f})")

# Coefficient comparison across models
print("\n\nCoefficient Comparison (scaled features):")
print("-" * 55)

for name, reg_model in models.items():
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    reg_model.fit(X_scaled, y)

    coef_df = pd.DataFrame({
        "Feature"    : X.columns,
        "Coefficient": reg_model.coef_
    }).sort_values("Coefficient", key=abs, ascending=False)

    print(f"\n{name}:")
    print(coef_df.to_string(index=False))

5-Fold Cross Validation Results:
---------------------------------------------

Linear Regression
  R²   : -2.2952 (± 2.7075)
  RMSE : 2375.300495 (± 1333.223707)

Ridge
  R²   : -2.2882 (± 2.7076)
  RMSE : 2371.581363 (± 1332.413070)

Lasso
  R²   : -2.2952 (± 2.7075)
  RMSE : 2375.300405 (± 1333.223692)


Coefficient Comparison (scaled features):
-------------------------------------------------------

Linear Regression:
        Feature  Coefficient
    day_of_week   208.774483
    # Following  -139.441901
 multiple_image   112.905767
sentiment_score     0.620682

Ridge:
        Feature  Coefficient
    day_of_week   206.406380
    # Following  -137.406798
 multiple_image   111.775376
sentiment_score     0.924131

Lasso:
        Feature  Coefficient
    day_of_week   208.774383
    # Following  -139.441758
 multiple_image   112.905663
sentiment_score     0.620597


In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error
import pandas as pd
import numpy as np

# 1. FEATURE ENGINEERING
df = df_merged.copy()

# Convert Day Posted to datetime and extract components
df["Day Posted"]  = pd.to_datetime(df["Day Posted"], format="%Y%m%d")
df["day_of_week"] = df["Day Posted"].dt.dayofweek   # 0=Monday, 6=Sunday
df["month"]       = df["Day Posted"].dt.month        # 1-12
df["year"]        = df["Day Posted"].dt.year         # e.g. 2026

# Convert Multiple Image Indicator to numeric
df["multiple_image"] = df["Multiple Image Indicator"].astype(int)

# Convert confidence to numeric (remove % sign)
df["confidence"] = df["confidence"].str.replace("%", "").astype(float)

# 2. DEFINE FEATURES AND TARGET
X = df[[
    "sentiment_score"
]]

y = df["# Comments"]

# 3. TRAIN / TEST SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 4. FIT REGRESSION
model = LinearRegression()
model.fit(X_train, y_train)

# 5. EVALUATE
y_pred = model.predict(X_test)
r2     = r2_score(y_test, y_pred)
rmse   = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"R² Score : {r2:.4f}")
print(f"RMSE     : {rmse:.6f}")

# 6. COEFFICIENTS — shows impact of each variable
coef_df = pd.DataFrame({
    "Feature"    : X.columns,
    "Coefficient": model.coef_
}).sort_values("Coefficient", ascending=False)

print("\nFeature Coefficients:")
print(coef_df.to_string(index=False))
print(f"\nIntercept: {model.intercept_:.6f}")

R² Score : -0.0307
RMSE     : 3331.760245

Feature Coefficients:
        Feature  Coefficient
sentiment_score   262.752277

Intercept: 1085.803279


In [ ]:
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score
import numpy as np

# Use cross-validation instead of a single
# train/test split — more reliable with 99 rows
models = {
    "Linear Regression": LinearRegression(),
    "Ridge"            : Ridge(alpha=1.0),
    "Lasso"            : Lasso(alpha=0.0001),
}

print("5-Fold Cross Validation Results:")
print("-" * 45)

for name, reg_model in models.items():
    # Scale features — important for Ridge and Lasso
    pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("model",  reg_model)
    ])

    cv_r2   = cross_val_score(pipeline, X, y, cv=5, scoring="r2")
    cv_rmse = cross_val_score(pipeline, X, y, cv=5,
                               scoring="neg_root_mean_squared_error")

    print(f"\n{name}")
    print(f"  R²   : {cv_r2.mean():.4f} (± {cv_r2.std():.4f})")
    print(f"  RMSE : {-cv_rmse.mean():.6f} (± {cv_rmse.std():.6f})")

# Coefficient comparison across models
print("\n\nCoefficient Comparison (scaled features):")
print("-" * 55)

for name, reg_model in models.items():
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    reg_model.fit(X_scaled, y)

    coef_df = pd.DataFrame({
        "Feature"    : X.columns,
        "Coefficient": reg_model.coef_
    }).sort_values("Coefficient", key=abs, ascending=False)

    print(f"\n{name}:")
    print(coef_df.to_string(index=False))

5-Fold Cross Validation Results:
---------------------------------------------

Linear Regression
  R²   : -2.0316 (± 2.8323)
  RMSE : 2182.494035 (± 1253.837805)

Ridge
  R²   : -2.0313 (± 2.8319)
  RMSE : 2182.403613 (± 1253.787903)

Lasso
  R²   : -2.0316 (± 2.8323)
  RMSE : 2182.494023 (± 1253.837804)


Coefficient Comparison (scaled features):
-------------------------------------------------------

Linear Regression:
        Feature  Coefficient
sentiment_score    31.483629

Ridge:
        Feature  Coefficient
sentiment_score    31.168793

Lasso:
        Feature  Coefficient
sentiment_score    31.483529


Predicting # Likes

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error
import pandas as pd
import numpy as np

# 1. FEATURE ENGINEERING
df = df_merged.copy()

# Convert Day Posted to datetime and extract components
df["Day Posted"]  = pd.to_datetime(df["Day Posted"], format="%Y%m%d")
df["day_of_week"] = df["Day Posted"].dt.dayofweek   # 0=Monday, 6=Sunday
df["month"]       = df["Day Posted"].dt.month        # 1-12
df["year"]        = df["Day Posted"].dt.year         # e.g. 2026

# Convert Multiple Image Indicator to numeric
df["multiple_image"] = df["Multiple Image Indicator"].astype(int)

# Convert confidence to numeric (remove % sign)
df["confidence"] = df["confidence"].str.replace("%", "").astype(float)

# 2. DEFINE FEATURES AND TARGET
X = df[[
    "sentiment_score",
    "multiple_image",
    "day_of_week",
    "month",
    "year",
    "# Following",
    "# Posts"
]]

y = df["# Likes"]

# 3. TRAIN / TEST SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 4. FIT REGRESSION
model = LinearRegression()
model.fit(X_train, y_train)

# 5. EVALUATE
y_pred = model.predict(X_test)
r2     = r2_score(y_test, y_pred)
rmse   = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"R² Score : {r2:.4f}")
print(f"RMSE     : {rmse:.6f}")

# 6. COEFFICIENTS — shows impact of each variable
coef_df = pd.DataFrame({
    "Feature"    : X.columns,
    "Coefficient": model.coef_
}).sort_values("Coefficient", ascending=False)

print("\nFeature Coefficients:")
print(coef_df.to_string(index=False))
print(f"\nIntercept: {model.intercept_:.6f}")

R² Score : -1.0792
RMSE     : 55522.017703

Feature Coefficients:
        Feature  Coefficient
sentiment_score 42614.295195
 multiple_image 29053.216524
           year  8488.577200
          month   774.644785
    # Following    91.553237
        # Posts    -0.183936
    day_of_week -5869.392003

Intercept: -17187293.913633


In [ ]:
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score
import numpy as np

# Use cross-validation instead of a single
# train/test split — more reliable with 99 rows
models = {
    "Linear Regression": LinearRegression(),
    "Ridge"            : Ridge(alpha=1.0),
    "Lasso"            : Lasso(alpha=0.0001),
}

print("5-Fold Cross Validation Results:")
print("-" * 45)

for name, reg_model in models.items():
    # Scale features — important for Ridge and Lasso
    pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("model",  reg_model)
    ])

    cv_r2   = cross_val_score(pipeline, X, y, cv=5, scoring="r2")
    cv_rmse = cross_val_score(pipeline, X, y, cv=5,
                               scoring="neg_root_mean_squared_error")

    print(f"\n{name}")
    print(f"  R²   : {cv_r2.mean():.4f} (± {cv_r2.std():.4f})")
    print(f"  RMSE : {-cv_rmse.mean():.6f} (± {cv_rmse.std():.6f})")

# Coefficient comparison across models
print("\n\nCoefficient Comparison (scaled features):")
print("-" * 55)

for name, reg_model in models.items():
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    reg_model.fit(X_scaled, y)

    coef_df = pd.DataFrame({
        "Feature"    : X.columns,
        "Coefficient": reg_model.coef_
    }).sort_values("Coefficient", key=abs, ascending=False)

    print(f"\n{name}:")
    print(coef_df.to_string(index=False))

5-Fold Cross Validation Results:
---------------------------------------------

Linear Regression
  R²   : -5.5300 (± 6.1271)
  RMSE : 66309.927210 (± 32968.721456)

Ridge
  R²   : -5.5771 (± 6.1610)
  RMSE : 66319.733379 (± 32862.128207)

Lasso
  R²   : -5.5300 (± 6.1271)
  RMSE : 66309.927332 (± 32968.721296)


Coefficient Comparison (scaled features):
-------------------------------------------------------

Linear Regression:
        Feature  Coefficient
sentiment_score 15370.567549
           year 11620.592489
 multiple_image 10425.906198
          month  9234.113650
    # Following  8972.482963
    day_of_week -3519.343912
        # Posts -2228.038172

Ridge:
        Feature  Coefficient
sentiment_score 15134.896876
           year 10826.762620
 multiple_image 10364.409802
    # Following  8956.838091
          month  8496.818024
    day_of_week -3452.338147
        # Posts -2045.073976

Lasso:
        Feature  Coefficient
sentiment_score 15370.567341
           year 11620.591626


In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error
import pandas as pd
import numpy as np

# 1. FEATURE ENGINEERING
df = df_merged.copy()

# Convert Day Posted to datetime and extract components
df["Day Posted"]  = pd.to_datetime(df["Day Posted"], format="%Y%m%d")
df["day_of_week"] = df["Day Posted"].dt.dayofweek   # 0=Monday, 6=Sunday
df["month"]       = df["Day Posted"].dt.month        # 1-12
df["year"]        = df["Day Posted"].dt.year         # e.g. 2026

# Convert Multiple Image Indicator to numeric
df["multiple_image"] = df["Multiple Image Indicator"].astype(int)

# Convert confidence to numeric (remove % sign)
df["confidence"] = df["confidence"].str.replace("%", "").astype(float)

# 2. DEFINE FEATURES AND TARGET
X = df[[
    "sentiment_score",
    "multiple_image",
    "day_of_week",
    "# Following"
]]

y = df["# Likes"]

# 3. TRAIN / TEST SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 4. FIT REGRESSION
model = LinearRegression()
model.fit(X_train, y_train)

# 5. EVALUATE
y_pred = model.predict(X_test)
r2     = r2_score(y_test, y_pred)
rmse   = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"R² Score : {r2:.4f}")
print(f"RMSE     : {rmse:.6f}")

# 6. COEFFICIENTS — shows impact of each variable
coef_df = pd.DataFrame({
    "Feature"    : X.columns,
    "Coefficient": model.coef_
}).sort_values("Coefficient", ascending=False)

print("\nFeature Coefficients:")
print(coef_df.to_string(index=False))
print(f"\nIntercept: {model.intercept_:.6f}")

R² Score : -1.0605
RMSE     : 55270.743780

Feature Coefficients:
        Feature  Coefficient
sentiment_score 40554.574232
 multiple_image 30665.183616
    # Following   100.019810
    day_of_week -6119.536120

Intercept: 3791.913499


In [ ]:
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score
import numpy as np

# Use cross-validation instead of a single
# train/test split — more reliable with 99 rows
models = {
    "Linear Regression": LinearRegression(),
    "Ridge"            : Ridge(alpha=1.0),
    "Lasso"            : Lasso(alpha=0.0001),
}

print("5-Fold Cross Validation Results:")
print("-" * 45)

for name, reg_model in models.items():
    # Scale features — important for Ridge and Lasso
    pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("model",  reg_model)
    ])

    cv_r2   = cross_val_score(pipeline, X, y, cv=5, scoring="r2")
    cv_rmse = cross_val_score(pipeline, X, y, cv=5,
                               scoring="neg_root_mean_squared_error")

    print(f"\n{name}")
    print(f"  R²   : {cv_r2.mean():.4f} (± {cv_r2.std():.4f})")
    print(f"  RMSE : {-cv_rmse.mean():.6f} (± {cv_rmse.std():.6f})")

# Coefficient comparison across models
print("\n\nCoefficient Comparison (scaled features):")
print("-" * 55)

for name, reg_model in models.items():
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    reg_model.fit(X_scaled, y)

    coef_df = pd.DataFrame({
        "Feature"    : X.columns,
        "Coefficient": reg_model.coef_
    }).sort_values("Coefficient", key=abs, ascending=False)

    print(f"\n{name}:")
    print(coef_df.to_string(index=False))

5-Fold Cross Validation Results:
---------------------------------------------

Linear Regression
  R²   : -4.0241 (± 4.4002)
  RMSE : 62641.869210 (± 34070.852784)

Ridge
  R²   : -3.9783 (± 4.3574)
  RMSE : 62514.857202 (± 34127.374516)

Lasso
  R²   : -4.0241 (± 4.4002)
  RMSE : 62641.869157 (± 34070.852804)


Coefficient Comparison (scaled features):
-------------------------------------------------------

Linear Regression:
        Feature  Coefficient
sentiment_score 14380.617529
 multiple_image 10707.972988
    # Following 10094.853435
    day_of_week -3790.593470

Ridge:
        Feature  Coefficient
sentiment_score 14233.574293
 multiple_image 10611.027858
    # Following 10000.612639
    day_of_week -3696.348272

Lasso:
        Feature  Coefficient
sentiment_score 14380.617417
 multiple_image 10707.972888
    # Following 10094.853328
    day_of_week -3790.593315


In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error
import pandas as pd
import numpy as np

# 1. FEATURE ENGINEERING
df = df_merged.copy()

# Convert Day Posted to datetime and extract components
df["Day Posted"]  = pd.to_datetime(df["Day Posted"], format="%Y%m%d")
df["day_of_week"] = df["Day Posted"].dt.dayofweek   # 0=Monday, 6=Sunday
df["month"]       = df["Day Posted"].dt.month        # 1-12
df["year"]        = df["Day Posted"].dt.year         # e.g. 2026

# Convert Multiple Image Indicator to numeric
df["multiple_image"] = df["Multiple Image Indicator"].astype(int)

# Convert confidence to numeric (remove % sign)
df["confidence"] = df["confidence"].str.replace("%", "").astype(float)

# 2. DEFINE FEATURES AND TARGET
X = df[[
    "sentiment_score"
]]

y = df["# Likes"]

# 3. TRAIN / TEST SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 4. FIT REGRESSION
model = LinearRegression()
model.fit(X_train, y_train)

# 5. EVALUATE
y_pred = model.predict(X_test)
r2     = r2_score(y_test, y_pred)
rmse   = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"R² Score : {r2:.4f}")
print(f"RMSE     : {rmse:.6f}")

# 6. COEFFICIENTS — shows impact of each variable
coef_df = pd.DataFrame({
    "Feature"    : X.columns,
    "Coefficient": model.coef_
}).sort_values("Coefficient", ascending=False)

print("\nFeature Coefficients:")
print(coef_df.to_string(index=False))
print(f"\nIntercept: {model.intercept_:.6f}")

R² Score : -0.2991
RMSE     : 43886.857588

Feature Coefficients:
        Feature  Coefficient
sentiment_score 45473.503643

Intercept: 21168.885246


In [ ]:
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score
import numpy as np

# Use cross-validation instead of a single
# train/test split — more reliable with 99 rows
models = {
    "Linear Regression": LinearRegression(),
    "Ridge"            : Ridge(alpha=1.0),
    "Lasso"            : Lasso(alpha=0.0001),
}

print("5-Fold Cross Validation Results:")
print("-" * 45)

for name, reg_model in models.items():
    # Scale features — important for Ridge and Lasso
    pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("model",  reg_model)
    ])

    cv_r2   = cross_val_score(pipeline, X, y, cv=5, scoring="r2")
    cv_rmse = cross_val_score(pipeline, X, y, cv=5,
                               scoring="neg_root_mean_squared_error")

    print(f"\n{name}")
    print(f"  R²   : {cv_r2.mean():.4f} (± {cv_r2.std():.4f})")
    print(f"  RMSE : {-cv_rmse.mean():.6f} (± {cv_rmse.std():.6f})")

# Coefficient comparison across models
print("\n\nCoefficient Comparison (scaled features):")
print("-" * 55)

for name, reg_model in models.items():
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    reg_model.fit(X_scaled, y)

    coef_df = pd.DataFrame({
        "Feature"    : X.columns,
        "Coefficient": reg_model.coef_
    }).sort_values("Coefficient", key=abs, ascending=False)

    print(f"\n{name}:")
    print(coef_df.to_string(index=False))

5-Fold Cross Validation Results:
---------------------------------------------

Linear Regression
  R²   : -3.1872 (± 4.0557)
  RMSE : 58155.287171 (± 35420.350680)

Ridge
  R²   : -3.1672 (± 4.0256)
  RMSE : 58122.474349 (± 35458.115773)

Lasso
  R²   : -3.1872 (± 4.0557)
  RMSE : 58155.287160 (± 35420.350703)


Coefficient Comparison (scaled features):
-------------------------------------------------------

Linear Regression:
        Feature  Coefficient
sentiment_score 15045.127647

Ridge:
        Feature  Coefficient
sentiment_score  14894.67637

Lasso:
        Feature  Coefficient
sentiment_score 15045.127547


Binary framing instead for engagement. Above average or below.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import numpy as np
import pandas as pd

# ─────────────────────────────────────────────
# 1. CREATE BINARY TARGET
#    1 = above average engagement
#    0 = below average engagement
# ─────────────────────────────────────────────
median_engagement = df["Engagement Calculation"].median()
df["high_engagement"] = (df["Engagement Calculation"] > median_engagement).astype(int)

print(f"Median engagement : {median_engagement:.6f}")
print(f"High engagement   : {df['high_engagement'].sum()} images")
print(f"Low engagement    : {(df['high_engagement'] == 0).sum()} images")

# ─────────────────────────────────────────────
# 2. FEATURES — same as best regression setup
# ─────────────────────────────────────────────
X = df[[
    "sentiment_score",
    "multiple_image",
    "day_of_week",
    "# Following"
]]
y = df["high_engagement"]

# ─────────────────────────────────────────────
# 3. MODELS
# ─────────────────────────────────────────────
models = {
    "Logistic Regression" : LogisticRegression(max_iter=1000),
    "Random Forest"       : RandomForestClassifier(n_estimators=100, random_state=42),
}

print("\n5-Fold Cross Validation Results:")
print("-" * 45)

for name, clf_model in models.items():
    pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("model",  clf_model)
    ])

    cv_acc = cross_val_score(pipeline, X, y, cv=5, scoring="accuracy")
    cv_f1  = cross_val_score(pipeline, X, y, cv=5, scoring="f1")

    print(f"\n{name}")
    print(f"  Accuracy : {cv_acc.mean():.4f} (± {cv_acc.std():.4f})")
    print(f"  F1 Score : {cv_f1.mean():.4f} (± {cv_f1.std():.4f})")

# ─────────────────────────────────────────────
# 4. FEATURE IMPORTANCE — sentiment vs others
# ─────────────────────────────────────────────
from sklearn.inspection import permutation_importance

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_scaled, y)

importance_df = pd.DataFrame({
    "Feature"   : X.columns,
    "Importance": rf.feature_importances_
}).sort_values("Importance", ascending=False)

print("\nFeature Importance (Random Forest):")
print(importance_df.to_string(index=False))

Median engagement : 0.001427
High engagement   : 49 images
Low engagement    : 50 images

5-Fold Cross Validation Results:
---------------------------------------------

Logistic Regression
  Accuracy : 0.5568 (± 0.1121)
  F1 Score : 0.4267 (± 0.1535)

Random Forest
  Accuracy : 0.4626 (± 0.1197)
  F1 Score : 0.4986 (± 0.1198)

Feature Importance (Random Forest):
        Feature  Importance
    # Following    0.499816
    day_of_week    0.345164
 multiple_image    0.084505
sentiment_score    0.070515
